### Blumenbeet – Lösung Schritt für Schritt

*38. Bundeswettbewerb Informatik (2019), 1. Runde, Aufgabe 1*

Im Blumenladen „Bunte Pracht“ sind kleine, rautenförmige Hochbeete der absolute Renner. Sie bestehen jeweils aus einer Kiste mit Erde, in die neun Blumenzwiebeln eingepflanzt werden.
Es gibt nur diese Blumenfarben: blau, gelb, grün, orange, rosa, rot und türkis.

Jeder Kunde gibt an, wie viele verschiedene Farben (höchstens 7) das Beet haben soll. Außerdem kann er Paare von Farben, die ihm nebeneinander besonders gut gefallen,
durch 1 bis 3 Bonuspunkte auszeichnen. Lea liebt rote neben blauen Blumen (3 Punkte), auch gefallen ihr rote neben türkisen Blumen (2 Punkte). Andere Kombinationen sind ihr egal.
Ein Beet, in dem zweimal rot neben türkis und einmal rot neben blau steht, erhält zum Beispiel 2 + 3 + 2 = 7 Punkte.

#### Aufgabe 1

Schreibe ein Programm, das die Angaben eines Kunden einliest und dann eine Bepflanzung des Hochbeetes errechnet, die die größtmögliche Punktsumme erzielt.
Als Nachbarblumen zählen dabei die Blumen links und rechts neben einer Blume sowie die bis zu vier Blumen schräg darunter oder darüber.
Im Beet aus der Aufgabe hat die rosa Blume vier Nachbarblumen, die blaue Blume sechs und die rote Blume ganz oben nur zwei.
Wende dein Programm auf alle Beispiele von Kundenangaben auf der BWINF-Website an.

#### Beispieldaten

- 1. Zeile: Anzahl der Farben (höchstens 7), 2. Zeile: Anzahl der Kundenwünsche (höchstens 7)
- je Wunsch eine Zeile: zwei Farben und die Bonuspunkte (1 bis 3)

`blumen.txt` (Leas Beispiel):

```
7
2
rot blau 3
rot tuerkis 2
```

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Brute Force (alles ausprobieren):** Es gibt nur 7⁹ = 40 353 607 Beete. Das ist für Python zu viel, aber der Ansatz „probiere alle Beete, nimm das beste“ ist der Ausgangspunkt.
- **Backtracking:** Wir pflanzen Platz für Platz und nehmen den letzten Schritt zurück, um eine andere Farbe zu probieren.
  Dabei kann man früh abbrechen, wenn ein Beet nicht mehr gültig oder nicht mehr gut genug werden kann.
- **Schranke (Branch and Bound):** Wir merken uns das beste Beet bisher. Kann der Rest eines angefangenen Beetes selbst im besten Fall nicht mehr Punkte bringen, probieren wir ihn gar nicht erst zu Ende.
- **Symmetrie nutzen:** Farben, die der Kunde in keinem Wunsch erwähnt, sind austauschbar. Es genügt, sie der Reihe nach zu verwenden.

**Nützliche Funktionen und Bibliotheken**
- `itertools.product(farben, repeat=9)` erzeugt alle Folgen aus 9 Farben (für den einfachen Brute-Force-Versuch).
- `set(beet)` zählt die verschiedenen Farben, `len(set(beet))`.
- Ein Dictionary mit dem Farbpaar als Schlüssel (`bonus[('rot', 'blau')]`) ist eine bequeme Wunschtabelle.

## Die Idee

1. **Das Beet modellieren.** Die neun Plätze liegen in den Reihen 1, 2, 3, 2, 1. Wir nummerieren sie von oben links nach unten rechts mit 0 bis 8 und legen für jeden Platz die Liste seiner Nachbarplätze an.
2. **Ein Beet bewerten.** Für jedes Paar benachbarter Plätze schlagen wir das Farbpaar in der Wunschtabelle nach und addieren die Punkte. Wichtig: Jedes Nachbarpaar zählt **einmal**, nicht zweimal.
3. **Alle Beete probieren,** aber klug: Nur Beete mit genau der gewünschten Zahl verschiedener Farben sind gültig. Und es lohnt sich nur, die Farben aus den Wünschen zu verwenden (plus ein paar beliebige „Füllfarben“,
   wenn der Kunde mehr Farben möchte, als er in den Wünschen nennt).
4. **Früh abbrechen,** wenn aus einem angefangenen Beet nichts Besseres mehr werden kann.

## Schritt 1: Einlesen

Wir lesen die Farbzahl und die Wünsche. Die Wünsche speichern wir in einem Dictionary `bonus`, und zwar für beide Reihenfolgen des Farbpaars (rot–blau und blau–rot), damit wir beim Nachschlagen nicht überlegen müssen, welche Farbe zuerst kommt.
Gibt ein Kunde für dasselbe Paar zwei verschiedene Werte an (`rot blau 3` und `blau rot 1`), ist das ein Widerspruch. Wir melden dann einen Fehler.

In [ ]:
FARBEN = ['blau', 'gelb', 'gruen', 'orange', 'rosa', 'rot', 'tuerkis']


def lies_wuensche(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = f.read().splitlines()
    farbanzahl = int(zeilen[0])
    anzahl_wuensche = int(zeilen[1])
    bonus = {}
    for zeile in zeilen[2:2 + anzahl_wuensche]:
        farbe1, farbe2, punkte = zeile.split()
        punkte = int(punkte)
        for paar in [(farbe1, farbe2), (farbe2, farbe1)]:
            if paar in bonus and bonus[paar] != punkte:
                raise ValueError(f'Widersprüchliche Wünsche für {farbe1} und {farbe2}')
            bonus[paar] = punkte
    return farbanzahl, bonus


farbanzahl, bonus = lies_wuensche('beispieldaten/blumen.txt')
print(farbanzahl)
print(bonus)

7
{('rot', 'blau'): 3, ('blau', 'rot'): 3, ('rot', 'tuerkis'): 2, ('tuerkis', 'rot'): 2}


## Schritt 2: Das Beet und seine Nachbarn

Wir geben jedem Platz eine Reihe `r` und eine waagerechte Position `x`. Die Plätze einer Reihe haben **Abstand 2** voneinander, die Reihen sind **um 1 versetzt**:

```
Reihe 0:             0
Reihe 1:         1       2
Reihe 2:     3       4       5
Reihe 3:         6       7
Reihe 4:             8
```

Zwei Plätze sind Nachbarn, wenn sie in derselben Reihe Abstand 2 haben (links und rechts) oder in benachbarten Reihen Abstand 1 (schräg darunter oder darüber).
Platz 4 (die Mitte) hat dann sechs Nachbarn, Platz 0 ganz oben nur zwei.

In [ ]:
X_JE_REIHE = [[0], [-1, 1], [-2, 0, 2], [-1, 1], [0]]   # waagerechte Positionen der Plätze in jeder Reihe

PLAETZE = []                                  # Platz i hat die Lage (Reihe, x)
for reihe in range(5):
    for x in X_JE_REIHE[reihe]:
        PLAETZE.append((reihe, x))


def sind_nachbarn(platz1, platz2):
    reihe1, x1 = platz1
    reihe2, x2 = platz2
    gleiche_reihe = reihe1 == reihe2 and abs(x1 - x2) == 2
    schraeg = abs(reihe1 - reihe2) == 1 and abs(x1 - x2) == 1
    return gleiche_reihe or schraeg


NACHBARN = []
for i in range(9):
    liste = []
    for j in range(9):
        if sind_nachbarn(PLAETZE[i], PLAETZE[j]):
            liste.append(j)
    NACHBARN.append(liste)

for i in range(9):
    print(i, NACHBARN[i])

0 [1, 2]
1 [0, 2, 3, 4]
2 [0, 1, 4, 5]
3 [1, 4, 6]
4 [1, 2, 3, 5, 6, 7]
5 [2, 4, 7]
6 [3, 4, 7, 8]
7 [4, 5, 6, 8]
8 [6, 7]


Platz 4 hat sechs Nachbarn, Platz 1 hat vier (0, 2, 3 und 4), die Ecken 0 und 8 haben je zwei. Das passt zur Aufgabe und zur Adjazenzmatrix in den Lösungshinweisen.

Damit wir Beete auch **sehen** können, schreiben wir eine kleine Ausgabefunktion, die die Raute zeichnet.

In [ ]:
def zeige_beet(beet):
    for reihe in range(5):
        zeile = [' '] * 27
        for platz in range(9):
            if PLAETZE[platz][0] == reihe:
                spalte = (PLAETZE[platz][1] + 2) * 5
                name = beet[platz]
                zeile[spalte:spalte + len(name)] = list(name)
        print(''.join(zeile).rstrip())


leas_beet = ['rot', 'tuerkis', 'rosa', 'gelb', 'blau', 'orange', 'gruen', 'rot', 'tuerkis']
zeige_beet(leas_beet)

          rot
     tuerkis   rosa
gelb      blau      orange
     gruen     rot
          tuerkis


## Schritt 3: Ein Beet bewerten

Für jeden Platz `i` gehen wir seine Nachbarn `j` durch. Damit jedes Nachbarpaar nur **einmal** zählt, nehmen wir nur Nachbarn mit `j > i`.
Paare ohne Wunsch bringen 0 Punkte (`bonus.get(paar, 0)`).

In [ ]:
def bewertung(beet, bonus):
    punkte = 0
    for i in range(9):
        for j in NACHBARN[i]:
            if j > i:                               # jedes Paar nur einmal
                punkte += bonus.get((beet[i], beet[j]), 0)
    return punkte


print(bewertung(leas_beet, bonus))

7


Das Beet aus der Aufgabe (`leas_beet`) bekommt 7 Punkte, wie im Aufgabentext. Zur Kontrolle von Hand: Rot liegt auf Platz 0 und 7. Platz 0 grenzt an 1 (türkis, +2) und 2 (rosa, 0). Platz 7 grenzt an 4 (blau, +3), 5 (orange), 6 (grün) und 8 (türkis, +2). Das sind 2 + 3 + 2 = 7 Punkte.

## Schritt 4: Brute Force für kleine Beispiele

Zuerst der ehrlich einfache Weg: alle Folgen aus 9 Farben erzeugen, nur die mit der richtigen Farbanzahl behalten und das Maximum suchen. Bei `blumen1.txt` wollen wir 2 Farben und es kommen
nur rot, türkis und grün in den Wünschen vor. Wir probieren darum **nur die Farben aus den Wünschen**: Mehr Auswahl kann nur dann helfen, wenn die Farbzahl nicht schon durch die Wunschfarben erreicht wird
(dazu gleich mehr). Das sind hier 3⁹ = 19 683 Beete.

In [ ]:
from itertools import product


def wunschfarben(bonus):
    return sorted({paar[0] for paar in bonus})


def brute_force(farbanzahl, bonus):
    farben = wunschfarben(bonus)
    bestes_beet = None
    beste_punkte = -1
    geprueft = 0
    for beet in product(farben, repeat=9):
        if len(set(beet)) != farbanzahl:             # falsche Zahl verschiedener Farben
            continue
        geprueft += 1
        punkte = bewertung(beet, bonus)
        if punkte > beste_punkte:
            beste_punkte = punkte
            bestes_beet = beet
    return beste_punkte, list(bestes_beet), geprueft


for nummer in [1, 2]:
    farbanzahl, bonus = lies_wuensche(f'beispieldaten/blumen{nummer}.txt')
    punkte, beet, geprueft = brute_force(farbanzahl, bonus)
    print(f'blumen{nummer}.txt: {punkte} Punkte, {geprueft} gültige Beete geprüft')
    print(beet)

blumen1.txt: 36 Punkte, 1530 gültige Beete geprüft
['rot', 'tuerkis', 'tuerkis', 'rot', 'rot', 'rot', 'tuerkis', 'tuerkis', 'rot']
blumen2.txt: 32 Punkte, 18150 gültige Beete geprüft
['gruen', 'rot', 'rot', 'tuerkis', 'tuerkis', 'tuerkis', 'rot', 'rot', 'tuerkis']


Das funktioniert, ist aber nur für diese kleinen Beispiele geeignet. Bei `blumen3.txt` bis `blumen5.txt` wollen wir 7 Farben, und von den 7⁹ ≈ 40 Millionen Beeten sind nur wenige gültig
(alle 7 Farben müssen vorkommen). Solche Beete einzeln zu erzeugen und zu prüfen, dauert in Python viel zu lange. Wir bauen das Beet daher **Platz für Platz** auf.

## Schritt 5: Backtracking mit Farbzählung und Füllfarben

**Welche Farben brauchen wir?** Ein Beet kann nur dann Punkte bringen, wenn zwei **Wunschfarben** nebeneinander liegen. Hat der Kunde `k` Wunschfarben, aber `f > k` Farben verlangt,
müssen wir die übrigen `f - k` Farben trotzdem irgendwo pflanzen. Welche das sind, ist egal, solange sie nicht in den Wünschen vorkommen. Wir nehmen einfach die ersten unbenutzten Farben aus der Liste `FARBEN`.
Mehr als `f` verschiedene Farben sind ohnehin nicht erlaubt.
Ist `f < k` (wie bei `blumen1.txt`), benutzen wir von den Wunschfarben nur `f` verschiedene.

**Austauschbare Füllfarben.** Füllfarben sind untereinander austauschbar. Wir erlauben deshalb die zweite Füllfarbe erst, wenn die erste schon vorkommt, und so weiter.
Das spart viele gleichwertige Beete.

**Abbruch.** Beim Pflanzen von Platz `i` zählen wir die verschiedenen Farben (`anzahl`). Wir brechen ab, wenn es schon mehr als `f` sind, oder wenn die übrigen Plätze nicht mehr reichen, um auf `f` zu kommen.

Die Punkte rechnen wir nicht jedes Mal neu: Beim Setzen von Platz `i` kommen nur die Punkte der Paare zwischen `i` und seinen **früheren** Nachbarn (`j < i`) dazu.

In [ ]:
def palette_bauen(farbanzahl, bonus):
    """Wunschfarben, danach so viele Füllfarben, dass es für farbanzahl Farben reicht."""
    palette = wunschfarben(bonus)
    anzahl_wunschfarben = len(palette)
    for farbe in FARBEN:
        if len(palette) < farbanzahl and farbe not in palette:
            palette.append(farbe)
    return palette, anzahl_wunschfarben


FRUEHERE = []                                  # frühere Nachbarn jedes Platzes
for i in range(9):
    FRUEHERE.append([j for j in NACHBARN[i] if j < i])


def suche(i, beet, punkte, benutzt, fuellfarben, status):
    """Setzt die Plätze i bis 8. benutzt: Farben im Beet, fuellfarben: Anzahl benutzter Füllfarben."""
    if i == 9:
        status['geprueft'] += 1
        if len(benutzt) == status['farbanzahl'] and punkte > status['beste_punkte']:
            status['beste_punkte'] = punkte
            status['bestes_beet'] = beet[:]
        return
    for nr in range(len(status['palette'])):
        if nr >= status['k'] and nr != status['k'] + fuellfarben:
            continue                              # Füllfarben nur der Reihe nach
        farbe = status['palette'][nr]
        neu = farbe not in benutzt
        anzahl = len(benutzt) + (1 if neu else 0)
        if anzahl > status['farbanzahl'] or anzahl + (8 - i) < status['farbanzahl']:
            continue                              # zu viele oder zu wenige verschiedene Farben
        zusatz = 0
        for j in FRUEHERE[i]:
            zusatz += status['bonus'].get((farbe, beet[j]), 0)
        beet[i] = farbe
        if neu:
            benutzt.add(farbe)
        suche(i + 1, beet, punkte + zusatz, benutzt, fuellfarben + (1 if neu and nr >= status['k'] else 0), status)
        if neu:
            benutzt.discard(farbe)


def loese_ohne_schranke(farbanzahl, bonus):
    palette, k = palette_bauen(farbanzahl, bonus)
    status = {'farbanzahl': farbanzahl, 'bonus': bonus, 'palette': palette, 'k': k,
              'geprueft': 0, 'beste_punkte': -1, 'bestes_beet': None}
    suche(0, [''] * 9, 0, set(), 0, status)
    return status['beste_punkte'], status['bestes_beet'], status['geprueft']


for nummer in [1, 2, 3]:
    farbanzahl, bonus = lies_wuensche(f'beispieldaten/blumen{nummer}.txt')
    punkte, beet, geprueft = loese_ohne_schranke(farbanzahl, bonus)
    print(f'blumen{nummer}.txt: {punkte} Punkte, {geprueft} Beete bis zum Ende gepflanzt')
    print(beet)

blumen1.txt: 36 Punkte, 1530 Beete bis zum Ende gepflanzt
['rot', 'tuerkis', 'tuerkis', 'rot', 'rot', 'rot', 'tuerkis', 'tuerkis', 'rot']
blumen2.txt: 32 Punkte, 18150 Beete bis zum Ende gepflanzt
['gruen', 'rot', 'rot', 'tuerkis', 'tuerkis', 'tuerkis', 'rot', 'rot', 'tuerkis']
blumen3.txt: 13 Punkte, 18900 Beete bis zum Ende gepflanzt
['gruen', 'rot', 'tuerkis', 'tuerkis', 'rot', 'blau', 'gelb', 'orange', 'rosa']


Bei `blumen1.txt` und `blumen2.txt` gibt es dieselben Punkte wie beim Brute Force (36 und 32). Bei `blumen3.txt` sind es 13 Punkte, mit rund 19 000 Beeten statt 40 Millionen.
Bei `blumen4.txt` und `blumen5.txt` ist die Lage schlechter: Dort kommen alle 7 Farben in den Wünschen vor, es gibt keine Füllfarben, die uns Beete sparen. Gültig sind alle Beete, in denen jede der 7 Farben mindestens einmal vorkommt,
und das sind 7!·S(9,7) = 2 328 480 (nach dem Ein- und Ausschließen: 7⁹ − 7·6⁹ + 21·5⁹ − …). Die Lösungshinweise nennen 10 372 320 Beete für diese Beispiele, weil ihr Verfahren weniger früh abbricht.
Mehr als zwei Millionen Beete in Python durchzuprobieren ist möglich, dauert aber schon spürbar. Die Schranke hilft hier am meisten.

## Schritt 6: Mit Schranke früher abbrechen

Wir merken uns die beste Punktzahl `beste_punkte`. Nach dem Setzen von Platz `i` stehen noch einige Nachbarpaare aus, nämlich alle Paare, deren späterer Platz größer als `i` ist.
Jedes bringt höchstens `maximum` Punkte, das ist der größte Bonus der Wünsche. Es gilt also

> Punkte jetzt + (Zahl der noch offenen Paare) · maximum ≥ Punkte, die das fertige Beet höchstens erreicht.

Ist das nicht größer als `beste_punkte`, kann das angefangene Beet das beste nicht mehr schlagen, und wir brechen ab. (Bei Gleichstand brechen wir auch ab, uns genügt *ein* bestes Beet.)

Die Zahl der offenen Paare ab Platz `i` (Paare `(i, j)` mit `j < i`, bei den Plätzen `i, i+1, ...`) berechnen wir einmal vorab in `OFFEN[i]`.

In [ ]:
OFFEN = []                                     # OFFEN[i]: Zahl der Nachbarpaare, die erst ab Platz i entstehen
for i in range(10):
    summe = 0
    for platz in range(i, 9):
        summe += len(FRUEHERE[platz])
    OFFEN.append(summe)
print(OFFEN)                                   # zusammen 16 Nachbarpaare im Beet


def suche_mit_schranke(i, beet, punkte, benutzt, fuellfarben, status):
    if i == 9:
        status['geprueft'] += 1
        if len(benutzt) == status['farbanzahl'] and punkte > status['beste_punkte']:
            status['beste_punkte'] = punkte
            status['bestes_beet'] = beet[:]
        return
    if punkte + OFFEN[i] * status['maximum'] <= status['beste_punkte']:
        return                                    # selbst im besten Fall nicht besser
    for nr in range(len(status['palette'])):
        if nr >= status['k'] and nr != status['k'] + fuellfarben:
            continue
        farbe = status['palette'][nr]
        neu = farbe not in benutzt
        anzahl = len(benutzt) + (1 if neu else 0)
        if anzahl > status['farbanzahl'] or anzahl + (8 - i) < status['farbanzahl']:
            continue
        zusatz = 0
        for j in FRUEHERE[i]:
            zusatz += status['bonus'].get((farbe, beet[j]), 0)
        beet[i] = farbe
        if neu:
            benutzt.add(farbe)
        suche_mit_schranke(i + 1, beet, punkte + zusatz, benutzt, fuellfarben + (1 if neu and nr >= status['k'] else 0), status)
        if neu:
            benutzt.discard(farbe)


def loese(farbanzahl, bonus):
    palette, k = palette_bauen(farbanzahl, bonus)
    status = {'farbanzahl': farbanzahl, 'bonus': bonus, 'palette': palette, 'k': k, 'maximum': max(bonus.values()),
              'geprueft': 0, 'beste_punkte': -1, 'bestes_beet': None}
    suche_mit_schranke(0, [''] * 9, 0, set(), 0, status)
    return status['beste_punkte'], status['bestes_beet'], status['geprueft']


for nummer in [1, 2, 3]:
    farbanzahl, bonus = lies_wuensche(f'beispieldaten/blumen{nummer}.txt')
    punkte, beet, geprueft = loese(farbanzahl, bonus)
    print(f'blumen{nummer}.txt: {punkte} Punkte, {geprueft} Beete bis zum Ende gepflanzt')

[16, 16, 15, 13, 12, 9, 7, 5, 2, 0]
blumen1.txt: 36 Punkte, 434 Beete bis zum Ende gepflanzt
blumen2.txt: 32 Punkte, 200 Beete bis zum Ende gepflanzt
blumen3.txt: 13 Punkte, 1990 Beete bis zum Ende gepflanzt


Die Schranke spart bei allen drei Beispielen noch einmal viel (bei `blumen2.txt` von 18 150 auf 200 Beete). Hier das Ergebnis für alle Beispiele (jetzt auch mit `blumen.txt`, `blumen4.txt` und `blumen5.txt`):

In [ ]:
for name in ['blumen', 'blumen1', 'blumen2', 'blumen3', 'blumen4', 'blumen5']:
    farbanzahl, bonus = lies_wuensche(f'beispieldaten/{name}.txt')
    punkte, beet, geprueft = loese(farbanzahl, bonus)
    print(f'{name}.txt: {punkte} Punkte, {geprueft} Beete bis zum Ende gepflanzt')
    print('  ', beet)
    assert bewertung(beet, bonus) == punkte and len(set(beet)) == farbanzahl

blumen.txt: 14 Punkte, 2309 Beete bis zum Ende gepflanzt
   ['blau', 'rot', 'rot', 'tuerkis', 'blau', 'gelb', 'gruen', 'orange', 'rosa']
blumen1.txt: 36 Punkte, 434 Beete bis zum Ende gepflanzt
   ['rot', 'tuerkis', 'tuerkis', 'rot', 'rot', 'rot', 'tuerkis', 'tuerkis', 'rot']
blumen2.txt: 32 Punkte, 200 Beete bis zum Ende gepflanzt
   ['gruen', 'rot', 'rot', 'tuerkis', 'tuerkis', 'tuerkis', 'rot', 'rot', 'tuerkis']
blumen3.txt: 13 Punkte, 1990 Beete bis zum Ende gepflanzt
   ['gruen', 'rot', 'tuerkis', 'tuerkis', 'rot', 'blau', 'gelb', 'orange', 'rosa']
blumen4.txt: 22 Punkte, 8872 Beete bis zum Ende gepflanzt
   ['blau', 'rosa', 'rosa', 'gruen', 'rot', 'rot', 'orange', 'tuerkis', 'gelb']
blumen5.txt: 22 Punkte, 21530 Beete bis zum Ende gepflanzt
   ['blau', 'rosa', 'rosa', 'gruen', 'rot', 'rot', 'orange', 'tuerkis', 'gelb']


Die Punktzahlen sind 14, 36, 32, 13, 22 und 22. Die Lösungshinweise nennen für `blumen1.txt` bis `blumen5.txt` genau 36, 32, 13, 22 und 22.
Unsere Beete sehen anders aus als die der Hinweise, denn es gibt oft mehrere beste Beete (zum Beispiel lassen sich zwei Farben vertauschen, wenn die Wünsche symmetrisch sind). Wir prüfen darum die **Punktzahl**,
nicht das Beet. Die Zahl der bis zum Ende gepflanzten Beete ist auch nicht dieselbe wie in den Hinweisen (dort 2 295, 19 674 und je 10 372 320), weil unsere Abbruchregeln anders sind.
Bei `blumen4.txt` und `blumen5.txt` sind es mit der Schranke 8 872 und 21 530 statt mehr als zwei Millionen. Bei `blumen.txt` (Leas Wünsche, 7 Farben) sind es 2 309; das beste Beet bringt 14 Punkte (die Hinweise nennen dafür keinen Wert).

Zum Schluss ein Beet als Raute (`blumen5.txt`):

In [ ]:
farbanzahl, bonus = lies_wuensche('beispieldaten/blumen5.txt')
punkte, beet, geprueft = loese(farbanzahl, bonus)
zeige_beet(beet)
print(punkte, 'Punkte')

          blau
     rosa      rosa
gruen     rot       rot
     orange    tuerkis
          gelb
22 Punkte


## Für Neugierige

- **Zweimal dasselbe Beet?** Wir haben nur die Füllfarben als austauschbar erkannt. Weitere Symmetrien gibt es auch: Spiegelt man das Beet an der senkrechten Achse, bleiben alle Nachbarschaften erhalten und die Punktzahl auch.
  Man könnte also die Hälfte der Beete weglassen (zum Beispiel verlangen, dass Platz 1 nicht „größer“ ist als Platz 2).
- **Bessere Schranken.** Statt jedem offenen Paar das Maximum zu geben, kann man für jede neu gepflanzte Blume den besten Bonus verwenden, den ihre Farbe mit irgendeiner anderen hat. Das schneidet früher ab.
- **Rechnen mit Reihen.** Das Beet hat nur fünf Reihen mit höchstens drei Blumen. Man kann zeilenweise vorgehen und für jede Belegung der letzten Reihe nur das beste Ergebnis davor merken (dynamische Programmierung). Das wäre auch mit viel größeren Beeten machbar.

## Zum Schluss: Blumenbeet mit MiniZinc

Die Aufgabe passt gut zu MiniZinc: Wir haben neun Entscheidungen (die Farben der Plätze), eine Bedingung (genau `f` verschiedene Farben) und eine Zielfunktion (die Punkte). Das Probieren übernimmt der Solver.

**Modell.**

- `x[i]` ist die **Farbe** (1 bis 7) auf Platz `i`.
- `kante` listet alle 16 Nachbarpaare auf, `w[a, b]` ist der Bonus für die Farben `a` und `b` nebeneinander (0, wenn kein Wunsch vorliegt).
- Die Punkte sind die Summe über alle Nachbarpaare. Dabei stehen `x[...]` als **Indizes** in der Tabelle `w`: `w[x[kante[e, 1]], x[kante[e, 2]]]`.
- Die Bedingung „genau `f` Farben“ zählt, für wie viele Farben `c` es einen Platz `i` mit `x[i] = c` gibt (`exists`).
- `solve maximize punkte` sucht die höchste Punktzahl.

```minizinc
int: f;                                     % gewünschte Zahl verschiedener Farben
int: m;                                     % Zahl der Nachbarpaare im Beet
array[1..m, 1..2] of 1..9: kante;           % kante[e, 1] und kante[e, 2] sind Nachbarplätze
array[1..7, 1..7] of int: w;                % w[a, b]: Bonus für die Farben a und b nebeneinander
array[1..7] of string: name;

array[1..9] of var 1..7: x;                 % x[i]: Farbe auf Platz i
constraint sum(c in 1..7)(bool2int(exists(i in 1..9)(x[i] = c))) = f;   % genau f Farben kommen vor

var int: punkte = sum(e in 1..m)(w[x[kante[e, 1]], x[kante[e, 2]]]);
solve maximize punkte;

output ["Punkte: ", show(punkte), "\n", show([name[fix(x[i])] | i in 1..9])];
```

**Daten.** Die Python-Zelle `als_dzn` baut die `.dzn`-Datei aus der Beispieldatei, indem sie die Kanten aus unserer Nachbarliste und die Wunschtabelle aus `bonus` holt. Für `blumen1.txt` sieht sie so aus:

```minizinc
f = 2;
m = 16;
kante = array2d(1..m, 1..2, [1, 2, 1, 3, 2, 3, 2, 4, 2, 5, 3, 5, 3, 6, 4, 5, 4, 7, 5, 6, 5, 7, 5, 8, 6, 8, 7, 8, 7, 9, 8, 9]);
w = array2d(1..7, 1..7, [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0]);
name = ["blau", "gelb", "gruen", "orange", "rosa", "rot", "tuerkis"];
```

Höchste Punktzahlen für alle sechs Dateien (Solver **Chuffed**, jede Datei braucht etwa eine Sekunde; Gecode schafft es in unter zwei Sekunden, CP-SAT braucht bis zu 25 Sekunden):

```
blumen.txt  ->  14 Punkte
blumen1.txt  ->  36 Punkte
blumen2.txt  ->  32 Punkte
blumen3.txt  ->  13 Punkte
blumen4.txt  ->  22 Punkte
blumen5.txt  ->  22 Punkte
```

Die Punktzahlen sind dieselben wie bei unserer Python-Lösung. Das Beet selbst kann sich unterscheiden, weil es oft mehrere beste Beete gibt. Das Modell ist deutlich kürzer als unser Backtracking,
und unsere Überlegungen zu Füllfarben und Schranken übernimmt der Solver selbst.

In [ ]:
def als_dzn(dateiname):
    farbanzahl, bonus = lies_wuensche(dateiname)
    kanten = []
    for i in range(9):
        for j in NACHBARN[i]:
            if j > i:
                kanten.append(f'{i + 1}, {j + 1}')
    tabelle = []
    for a in FARBEN:
        for b in FARBEN:
            tabelle.append(str(bonus.get((a, b), 0)))
    namen = ', '.join(chr(34) + f + chr(34) for f in FARBEN)
    return (f'f = {farbanzahl};\nm = {len(kanten)};\nkante = array2d(1..m, 1..2, [{", ".join(kanten)}]);\n'
            f'w = array2d(1..7, 1..7, [{", ".join(tabelle)}]);\nname = [{namen}];')


print(als_dzn('beispieldaten/blumen1.txt'))

f = 2;
m = 16;
kante = array2d(1..m, 1..2, [1, 2, 1, 3, 2, 3, 2, 4, 2, 5, 3, 5, 3, 6, 4, 5, 4, 7, 5, 6, 5, 7, 5, 8, 6, 8, 7, 8, 7, 9, 8, 9]);
w = array2d(1..7, 1..7, [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0]);
name = ["blau", "gelb", "gruen", "orange", "rosa", "rot", "tuerkis"];
